## Cell 1 — Startup Smoke Test
### Quick pre-training check
Run the next cell first to verify Kaggle environment + FaceForensics++ dataset structure before starting full training.

In [ ]:
# Cell 2 — Smoke Test (Pre-Training Readiness)
# Quick check for Kaggle runtime, dataset mount, and FF++ folder structure.

from pathlib import Path
import os
import json

SEARCH_ROOTS = [Path('/kaggle/input'), Path('/kaggle/working')]
RUN_ON_KAGGLE = Path('/kaggle/input').exists()


def looks_like_ffpp_root(path: Path) -> bool:
    return (path / 'original_sequences').exists() and (path / 'manipulated_sequences').exists()


def find_ffpp_candidates(roots):
    candidates = []
    seen = set()

    for root in roots:
        if not root.exists():
            continue
        for top in root.iterdir():
            if not top.is_dir():
                continue
            if looks_like_ffpp_root(top):
                key = str(top.resolve())
                if key not in seen:
                    seen.add(key)
                    candidates.append(top)
            for child in top.iterdir():
                if child.is_dir() and looks_like_ffpp_root(child):
                    key = str(child.resolve())
                    if key not in seen:
                        seen.add(key)
                        candidates.append(child)
    return candidates


ffpp_candidates = find_ffpp_candidates(SEARCH_ROOTS)
selected_root = ffpp_candidates[0] if ffpp_candidates else None

compression_found = None
real_count = 0
method_counts = {}

if selected_root is not None:
    for comp in ['c23', 'c40', 'raw']:
        real_dir = selected_root / 'original_sequences' / 'youtube' / comp / 'videos'
        if real_dir.exists():
            compression_found = comp
            real_count = len(list(real_dir.glob('*.mp4')))
            break

    manip_root = selected_root / 'manipulated_sequences'
    if compression_found and manip_root.exists():
        for method_dir in manip_root.iterdir():
            if not method_dir.is_dir():
                continue
            vid_dir = method_dir / compression_found / 'videos'
            if vid_dir.exists():
                method_counts[method_dir.name] = len(list(vid_dir.glob('*.mp4')))

status = {
    'run_on_kaggle': RUN_ON_KAGGLE,
    'python_version': os.sys.version.split()[0],
    'search_roots': [str(p) for p in SEARCH_ROOTS if p.exists()],
    'ffpp_root_detected': str(selected_root) if selected_root else None,
    'compression_found': compression_found,
    'real_videos_found': int(real_count),
    'fake_methods_found': int(len(method_counts)),
    'ready_for_training': bool(selected_root and compression_found and real_count > 0 and len(method_counts) > 0)
}

print('=== Smoke Test Status ===')
print(json.dumps(status, indent=2))

if ffpp_candidates:
    print('\nDetected candidate roots:')
    for candidate in ffpp_candidates[:10]:
        print(' -', candidate)

if method_counts:
    print('\nTop fake methods by video count:')
    for k, v in sorted(method_counts.items(), key=lambda x: x[1], reverse=True)[:8]:
        print(f'  {k}: {v}')

if not status['ready_for_training']:
    print('\n[Action Required] Dataset mount/path is not fully ready. If you use the official script, run Cell 4 first, then rerun this smoke test.')
else:
    print('\n[OK] Environment and dataset structure look ready for training.')

# DeepFake Detection (Kaggle, Full FaceForensics++)
This notebook is a **binary detector** (`deepfake` vs `real`) trained on full FaceForensics++ videos using a **dual-stream frame model**:
- **Global branch**: semantic frame features from RGB
- **Local branch**: artifact-focused features from high-frequency + edge + SegFormer uncertainty maps
- **Fusion**: local/global features are fused per frame, then aggregated temporally
Exports checkpoints as **`.pth`** (no `.pkl`).

In [ ]:
# Cell 4 — Configure and optionally run the official FF++ download script
# Leave RUN_FFPP_DOWNLOAD_SCRIPT=False unless you already have approved access and the script file.

import os
import sys
import shlex
import shutil
import subprocess
from pathlib import Path

RUN_FFPP_DOWNLOAD_SCRIPT = False

# Put the approved script in one of these places, then set the correct path below.
FFPP_SCRIPT_CANDIDATES = [
    Path('/kaggle/working/download_faceforensics.py'),
    Path('/kaggle/working/download_ffpp.py'),
    Path('/kaggle/input/ffpp-script/download_faceforensics.py'),
    Path('/kaggle/input/ffpp-script/download_ffpp.py')
]
FFPP_SCRIPT_PATH = next((p for p in FFPP_SCRIPT_CANDIDATES if p.exists()), FFPP_SCRIPT_CANDIDATES[0])

# Choose where downloaded files should be written.
FFPP_DOWNLOAD_DIR = Path('/kaggle/working/faceforensicspp')
FFPP_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

# Example only. Replace with the exact arguments required by the official script you receive.
# Common examples may include compression, methods, dataset type, output dir, credentials, or metadata csv.
FFPP_SCRIPT_ARGS = [
    '--output_dir', str(FFPP_DOWNLOAD_DIR),
    '--compression', 'c23'
]

# Optional secrets/credentials if the official script requires them.
# In Kaggle, prefer Add-ons > Secrets, then expose them as env vars.
OPTIONAL_SECRET_ENV_VARS = [
    'FFPP_USERNAME',
    'FFPP_PASSWORD',
    'FFPP_TOKEN',
    'FFPP_EMAIL'
]

print('RUN_FFPP_DOWNLOAD_SCRIPT =', RUN_FFPP_DOWNLOAD_SCRIPT)
print('Resolved script path =', FFPP_SCRIPT_PATH)
print('Download dir =', FFPP_DOWNLOAD_DIR)
print('Available secret env vars =', {k: bool(os.environ.get(k)) for k in OPTIONAL_SECRET_ENV_VARS})

if RUN_FFPP_DOWNLOAD_SCRIPT:
    if not FFPP_SCRIPT_PATH.exists():
        raise FileNotFoundError(
            f'Official FF++ script not found at {FFPP_SCRIPT_PATH}. '
            'Upload the approved script to /kaggle/working or attach it as a Kaggle dataset.'
        )

    command = [sys.executable, str(FFPP_SCRIPT_PATH), *FFPP_SCRIPT_ARGS]
    print('Running command:', ' '.join(shlex.quote(x) for x in command))

    result = subprocess.run(
        command,
        cwd=str(FFPP_SCRIPT_PATH.parent),
        capture_output=True,
        text=True,
        check=False
    )

    print('\n=== SCRIPT STDOUT ===')
    print(result.stdout[-12000:] if result.stdout else '[no stdout]')
    print('\n=== SCRIPT STDERR ===')
    print(result.stderr[-12000:] if result.stderr else '[no stderr]')
    print('\nExit code =', result.returncode)

    if result.returncode != 0:
        raise RuntimeError(
            'Official FF++ script failed. Check the required arguments, credentials, '
            'internet setting in Kaggle, and the script documentation you received.'
        )

    print('\n[OK] Official FF++ script completed.')
else:
    print('Skipped script execution. Set RUN_FFPP_DOWNLOAD_SCRIPT=True after you receive the approved script and arguments.')

## Cell 3 — Optional official FF++ download script
### Use this only after approval
If the FaceForensics++ team gives you an official download script, place it in Kaggle working storage or attach it as a dataset, then configure the next cell to run it before training.

## Restructured Prompt + TODO
### Corrected objective
Train a Kaggle GPU-ready, full FaceForensics++ **binary** detector that uses **parallel local and global per-frame features**, fuses them, aggregates over time, and reports strong evidence metrics (AUC/F1/EER/confusion matrix).
### Research-informed TODO
1. Use full FF++ sources (all real + all manipulation methods) with group-safe train/val/test split.
2. Build artifact maps per frame (frequency + edges + optional SegFormer uncertainty).
3. Extract global and local features in parallel and fuse at frame level.
4. Add temporal aggregation (GRU) and binary classification head.
5. Train with class balancing, AMP, scheduler, early stopping.
6. Report Accuracy, Precision, Recall, F1, ROC-AUC, PR-AUC, EER, confusion matrix.
7. Export `.pth` model + config + threshold + metrics for project compatibility.
### Research references behind this design
- FaceForensics++ benchmark (RÃƒÂ¶ssler et al., 2019)
- SegFormer (Xie et al., 2021) for multiscale local/global segmentation cues
- Multi-attentional Deepfake Detection (Zhao et al., 2021) for local artifact emphasis
- F3-Net / frequency cues (Qian et al., 2020) for compression-robust forgery traces

In [ ]:
# Optional installs (Kaggle usually has most of these)
# !pip -q install -U timm transformers scikit-learn

In [ ]:
import os, json, math, random
from pathlib import Path
from dataclasses import dataclass
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix, roc_curve
)
SEED = 42
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

In [ ]:
# ===== Config =====
KAGGLE_INPUT_ROOT = Path('/kaggle/input')
KAGGLE_WORKING_ROOT = Path('/kaggle/working')
RUN_ON_KAGGLE = KAGGLE_INPUT_ROOT.exists()


def _looks_like_ffpp_root(path: Path) -> bool:
    return (path / 'original_sequences').exists() and (path / 'manipulated_sequences').exists()


def discover_ffpp_root(preferred: Path) -> Path:
    if _looks_like_ffpp_root(preferred):
        return preferred

    candidates = [
        Path('/kaggle/input/faceforensicsplusplus'),
        Path('/kaggle/input/faceforensics-plus-plus'),
        Path('/kaggle/input/faceforensics'),
        Path('/kaggle/input/ffpp'),
        Path('/kaggle/working/faceforensicsplusplus'),
        Path('/kaggle/working/faceforensics-plus-plus'),
        Path('/kaggle/working/faceforensics'),
        Path('/kaggle/working/ffpp'),
        Path('/kaggle/working/faceforensicspp')
    ]

    for candidate in candidates:
        if _looks_like_ffpp_root(candidate):
            return candidate

    scan_roots = []
    if RUN_ON_KAGGLE and KAGGLE_INPUT_ROOT.exists():
        scan_roots.append(KAGGLE_INPUT_ROOT)
    if KAGGLE_WORKING_ROOT.exists():
        scan_roots.append(KAGGLE_WORKING_ROOT)

    for scan_root in scan_roots:
        scan_pool = []
        for top in scan_root.iterdir():
            if not top.is_dir():
                continue
            scan_pool.append(top)
            for child in top.iterdir():
                if child.is_dir():
                    scan_pool.append(child)

        for candidate in scan_pool:
            if _looks_like_ffpp_root(candidate):
                return candidate

    return preferred

CFG = {
    'ffpp_root': Path('/kaggle/input/faceforensicsplusplus'),
    'compression': 'c23',     # one of: raw, c23, c40
    'frames_per_video': 12,
    'frame_size': 224,
    'batch_size': 6,
    'num_workers': 2,
    'epochs': 12,
    'lr': 2e-4,
    'weight_decay': 1e-4,
    'patience': 3,
    'max_videos_per_class': None,  # None = full dataset
    'fake_methods': ['Deepfakes', 'Face2Face', 'FaceSwap', 'NeuralTextures'],
    'save_dir': Path('/kaggle/working') if RUN_ON_KAGGLE else Path.cwd(),
    'model_name': 'ffpp_dualstream_segformer_binary.pth'
}

CFG['ffpp_root'] = discover_ffpp_root(CFG['ffpp_root'])
CFG['save_dir'].mkdir(parents=True, exist_ok=True)

print('RUN_ON_KAGGLE:', RUN_ON_KAGGLE)
print('Resolved FF++ root:', CFG['ffpp_root'])
print('Save dir:', CFG['save_dir'])
CFG

In [ ]:
# ===== FaceForensics++ full-data indexing =====
VIDEO_EXTS = ('.mp4', '.avi', '.mov', '.mkv')

def _list_videos(folder: Path):
    if not folder.exists():
        return []
    out = []
    for pattern in VIDEO_EXTS:
        out.extend(sorted(folder.glob(f'*{pattern}')))
    return out

def infer_video_group_id(p: Path):
    stem = p.stem
    if '_' in stem and all(x.isdigit() for x in stem.split('_') if x):
        return stem
    return stem

def resolve_compression(root: Path, preferred: str):
    real_base = root / 'original_sequences' / 'youtube'
    for comp in [preferred, 'c23', 'c40', 'raw']:
        if (real_base / comp / 'videos').exists():
            return comp
    return preferred

def resolve_method_dir(root: Path, method: str, compression: str) -> Path:
    manip_root = root / 'manipulated_sequences'

    # 1) exact name
    candidate = manip_root / method / compression / 'videos'
    if candidate.exists():
        return candidate

    # 2) case-insensitive match
    if manip_root.exists():
        for folder in manip_root.iterdir():
            if folder.is_dir() and folder.name.lower() == method.lower():
                candidate = folder / compression / 'videos'
                if candidate.exists():
                    return candidate

    # 3) known aliases
    aliases = {
        'deepfakes': ['Deepfakes', 'DeepFakeDetection', 'deepfakes'],
        'face2face': ['Face2Face', 'face2face'],
        'faceswap': ['FaceSwap', 'faceswap'],
        'neuraltextures': ['NeuralTextures', 'neuraltextures']
    }

    for alias in aliases.get(method.lower(), [method]):
        candidate = manip_root / alias / compression / 'videos'
        if candidate.exists():
            return candidate

    return manip_root / method / compression / 'videos'

def build_ffpp_binary_index(ffpp_root: Path, compression='c23', fake_methods=None, max_videos_per_class=None):
    if fake_methods is None:
        fake_methods = ['Deepfakes', 'Face2Face', 'FaceSwap', 'NeuralTextures']

    ffpp_root = Path(ffpp_root)
    if not ffpp_root.exists():
        raise RuntimeError(f'FF++ root does not exist: {ffpp_root}')

    compression = resolve_compression(ffpp_root, compression)
    print('Using compression:', compression)

    rows = []

    # Real videos
    real_dir = ffpp_root / 'original_sequences' / 'youtube' / compression / 'videos'
    real_videos = _list_videos(real_dir)
    if max_videos_per_class is not None:
        real_videos = real_videos[:max_videos_per_class]

    for vp in real_videos:
        rows.append({
            'video_path': str(vp),
            'label': 0,
            'label_name': 'real',
            'method': 'youtube',
            'group_id': infer_video_group_id(vp)
        })

    # Fake videos (all methods => binary fake=1)
    for method in fake_methods:
        fake_dir = resolve_method_dir(ffpp_root, method, compression)
        fake_videos = _list_videos(fake_dir)
        if max_videos_per_class is not None:
            fake_videos = fake_videos[:max_videos_per_class]

        print(f'Method {method}: {len(fake_videos)} videos')
        for vp in fake_videos:
            rows.append({
                'video_path': str(vp),
                'label': 1,
                'label_name': 'deepfake',
                'method': method,
                'group_id': infer_video_group_id(vp)
            })

    df = pd.DataFrame(rows)
    if len(df) == 0:
        raise RuntimeError(
            f'No videos found under {ffpp_root}. '
            f'Check dataset mount and directory structure for FaceForensics++.'
        )

    print('Total videos:', len(df))
    print('Label counts:\n', df['label_name'].value_counts())
    print('Method counts:\n', df['method'].value_counts())
    return df

df_all = build_ffpp_binary_index(
    CFG['ffpp_root'],
    compression=CFG['compression'],
    fake_methods=CFG['fake_methods'],
    max_videos_per_class=CFG['max_videos_per_class']
)
df_all.head()

In [ ]:
# ===== Group-safe split (prevents leakage by video identity) =====
def split_df_grouped(df, train_size=0.7, val_size=0.15, test_size=0.15, random_state=SEED):
    assert abs(train_size + val_size + test_size - 1.0) < 1e-6
    gss_1 = GroupShuffleSplit(n_splits=1, train_size=train_size, random_state=random_state)
    idx_train, idx_temp = next(gss_1.split(df, y=df['label'], groups=df['group_id']))
    train_df = df.iloc[idx_train].reset_index(drop=True)
    temp_df = df.iloc[idx_temp].reset_index(drop=True)
    rel_val = val_size / (val_size + test_size)
    gss_2 = GroupShuffleSplit(n_splits=1, train_size=rel_val, random_state=random_state + 1)
    idx_val, idx_test = next(gss_2.split(temp_df, y=temp_df['label'], groups=temp_df['group_id']))
    val_df = temp_df.iloc[idx_val].reset_index(drop=True)
    test_df = temp_df.iloc[idx_test].reset_index(drop=True)
    return train_df, val_df, test_df
train_df, val_df, test_df = split_df_grouped(df_all)
print('train/val/test:', len(train_df), len(val_df), len(test_df))
print('Train labels:\n', train_df['label_name'].value_counts())

In [ ]:
# ===== SegFormer helper (optional, fallback-safe) =====
class SegFormerUncertaintyExtractor:
    def __init__(self, device=DEVICE):
        self.device = device
        self.ready = False
        self.processor = None
        self.model = None
        try:
            from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation
            self.processor = SegformerImageProcessor.from_pretrained('nvidia/segformer-b0-finetuned-ade-512-512')
            self.model = SegformerForSemanticSegmentation.from_pretrained('nvidia/segformer-b0-finetuned-ade-512-512').to(device).eval()
            self.ready = True
            print('SegFormer loaded from transformers.')
        except Exception as e:
            print('[WARN] SegFormer unavailable, using edge/frequency fallback only:', str(e)[:140])
    @torch.no_grad()
    def uncertainty_map(self, rgb_u8):
        h, w = rgb_u8.shape[:2]
        if not self.ready:
            return np.zeros((h, w), dtype=np.float32)
        inputs = self.processor(images=rgb_u8, return_tensors='pt').to(self.device)
        logits = self.model(**inputs).logits
        probs = torch.softmax(logits, dim=1)
        entropy = -(probs * torch.log(probs + 1e-8)).sum(dim=1, keepdim=True)
        ent = F.interpolate(entropy, size=(h, w), mode='bilinear', align_corners=False)[0,0].detach().cpu().numpy()
        ent = (ent - ent.min()) / (ent.max() - ent.min() + 1e-8)
        return ent.astype(np.float32)
seg_uncert = SegFormerUncertaintyExtractor(device=DEVICE)

In [ ]:
# ===== Dataset with per-frame local/global parallel inputs =====
def sample_even_frames(video_path, n_frames=12):
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return []
    idxs = np.linspace(0, max(total-1, 0), n_frames).astype(int)
    frames = []
    for idx in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, fr = cap.read()
        if ok and fr is not None:
            frames.append(fr)
    cap.release()
    return frames
def build_local_artifact_map(bgr, segformer_extractor=None):
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
    lap = cv2.Laplacian(gray, cv2.CV_32F, ksize=3)
    lap = np.abs(lap)
    lap = (lap - lap.min()) / (lap.max() - lap.min() + 1e-8)
    dct = cv2.dct(gray)
    h, w = dct.shape
    y, x = np.indices((h, w))
    high_mask = ((x + y) > (0.35 * (h + w))).astype(np.float32)
    high = np.abs(dct) * high_mask
    high = cv2.GaussianBlur(high, (7,7), 0)
    high = (high - high.min()) / (high.max() - high.min() + 1e-8)
    seg_u = np.zeros_like(gray, dtype=np.float32)
    if segformer_extractor is not None:
        seg_u = segformer_extractor.uncertainty_map(rgb)
    artifact = 0.40 * lap + 0.35 * high + 0.25 * seg_u
    artifact = np.clip(artifact, 0.0, 1.0).astype(np.float32)
    return artifact
class FFPPDualStreamDataset(Dataset):
    def __init__(self, df, frames_per_video=12, frame_size=224, segformer_extractor=None):
        self.df = df.reset_index(drop=True)
        self.frames_per_video = frames_per_video
        self.frame_size = frame_size
        self.segformer_extractor = segformer_extractor
    def __len__(self):
        return len(self.df)
    def _prep_rgb(self, bgr):
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        rgb = cv2.resize(rgb, (self.frame_size, self.frame_size), interpolation=cv2.INTER_AREA)
        x = torch.from_numpy(rgb).permute(2,0,1).float() / 255.0
        x = (x - torch.tensor([0.485,0.456,0.406]).view(3,1,1)) / torch.tensor([0.229,0.224,0.225]).view(3,1,1)
        return x
    def _prep_local(self, bgr):
        art = build_local_artifact_map(bgr, self.segformer_extractor)
        art = cv2.resize(art, (self.frame_size, self.frame_size), interpolation=cv2.INTER_AREA)
        return torch.from_numpy(art).unsqueeze(0).float()
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        video_path = row['video_path']
        y = int(row['label'])
        frames = sample_even_frames(video_path, n_frames=self.frames_per_video)
        if len(frames) == 0:
            frames = [np.zeros((self.frame_size, self.frame_size, 3), dtype=np.uint8)] * self.frames_per_video
        if len(frames) < self.frames_per_video:
            frames.extend([frames[-1]] * (self.frames_per_video - len(frames)))
        else:
            frames = frames[:self.frames_per_video]
        x_global = torch.stack([self._prep_rgb(fr) for fr in frames], dim=0)   # (T,3,H,W)
        x_local = torch.stack([self._prep_local(fr) for fr in frames], dim=0)  # (T,1,H,W)
        return {
            'x_global': x_global,
            'x_local': x_local,
            'label': torch.tensor(y, dtype=torch.float32),
            'video_path': video_path
        }

In [ ]:
# ===== Dual-stream model: local/global parallel then converge =====
class LocalBranch(nn.Module):
    def __init__(self, out_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1)
        )
        self.fc = nn.Linear(128, out_dim)
    def forward(self, x):
        x = self.net(x).flatten(1)
        return self.fc(x)
class GlobalBranch(nn.Module):
    def __init__(self, out_dim=256):
        super().__init__()
        backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        self.features = nn.Sequential(*list(backbone.children())[:-1])
        self.fc = nn.Linear(512, out_dim)
    def forward(self, x):
        x = self.features(x).flatten(1)
        return self.fc(x)
class DualStreamDeepfakeNet(nn.Module):
    def __init__(self, feat_dim=256, fusion_dim=256, rnn_hidden=256, rnn_layers=1, dropout=0.25):
        super().__init__()
        self.local_branch = LocalBranch(out_dim=feat_dim)
        self.global_branch = GlobalBranch(out_dim=feat_dim)
        self.fusion = nn.Sequential(
            nn.Linear(feat_dim * 2, fusion_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout)
        )
        self.temporal = nn.GRU(
            input_size=fusion_dim,
            hidden_size=rnn_hidden,
            num_layers=rnn_layers,
            batch_first=True,
            dropout=dropout if rnn_layers > 1 else 0.0
        )
        self.head = nn.Sequential(
            nn.Linear(rnn_hidden, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
        )
    def forward(self, x_global, x_local):
        # x_global: (B,T,3,H,W), x_local: (B,T,1,H,W)
        b, t, _, _, _ = x_global.shape
        g = x_global.reshape(b*t, *x_global.shape[2:])
        l = x_local.reshape(b*t, *x_local.shape[2:])
        g_feat = self.global_branch(g)
        l_feat = self.local_branch(l)
        fused = self.fusion(torch.cat([g_feat, l_feat], dim=1))
        fused = fused.reshape(b, t, -1)
        _, h = self.temporal(fused)
        h_last = h[-1]
        logits = self.head(h_last).squeeze(1)
        return logits

In [ ]:
# ===== Train / Eval helpers =====
@torch.no_grad()
def evaluate_model(model, loader, device=DEVICE):
    model.eval()
    all_probs, all_true = [], []
    for batch in loader:
        xg = batch['x_global'].to(device)
        xl = batch['x_local'].to(device)
        y = batch['label'].cpu().numpy().astype(int)
        logits = model(xg, xl)
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        all_probs.extend(probs.tolist())
        all_true.extend(y.tolist())
    y_true = np.array(all_true, dtype=np.int64)
    y_prob = np.array(all_probs, dtype=np.float32)
    y_pred = (y_prob >= 0.5).astype(np.int64)
    metrics = {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
        'roc_auc': float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) > 1 else None,
        'pr_auc': float(average_precision_score(y_true, y_prob)) if len(np.unique(y_true)) > 1 else None,
        'confusion_matrix': confusion_matrix(y_true, y_pred).tolist()
    }
    if len(np.unique(y_true)) > 1:
        fpr, tpr, thr = roc_curve(y_true, y_prob)
        fnr = 1.0 - tpr
        idx = np.nanargmin(np.abs(fnr - fpr))
        metrics['eer'] = float((fpr[idx] + fnr[idx]) / 2.0)
        metrics['eer_threshold'] = float(thr[idx])
    else:
        metrics['eer'] = None
        metrics['eer_threshold'] = None
    return metrics, y_true, y_prob
def train_one_run(model, train_loader, val_loader, cfg, device=DEVICE):
    pos = int((train_df['label'] == 1).sum())
    neg = int((train_df['label'] == 0).sum())
    pos_weight = torch.tensor([max(neg,1)/max(pos,1)], device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg['epochs'])
    scaler = torch.cuda.amp.GradScaler(enabled=(device=='cuda'))
    best = {'f1': -1.0, 'state': None, 'epoch': -1, 'metrics': None}
    wait = 0
    history = []
    for epoch in range(1, cfg['epochs'] + 1):
        model.train()
        epoch_loss = 0.0
        for batch in train_loader:
            xg = batch['x_global'].to(device)
            xl = batch['x_local'].to(device)
            y = batch['label'].to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(device=='cuda')):
                logits = model(xg, xl)
                loss = criterion(logits, y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            epoch_loss += float(loss.item())
        scheduler.step()
        val_metrics, _, _ = evaluate_model(model, val_loader, device=device)
        row = {'epoch': epoch, 'train_loss': epoch_loss / max(len(train_loader),1), **val_metrics}
        history.append(row)
        print(f"Epoch {epoch}/{cfg['epochs']} | loss={row['train_loss']:.4f} | f1={row['f1']:.4f} | auc={row['roc_auc']}")
        if row['f1'] > best['f1']:
            best['f1'] = row['f1']
            best['state'] = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            best['epoch'] = epoch
            best['metrics'] = val_metrics
            wait = 0
        else:
            wait += 1
        if wait >= cfg['patience']:
            print(f'Early stopping at epoch {epoch}')
            break
    if best['state'] is not None:
        model.load_state_dict(best['state'])
    return model, pd.DataFrame(history), best

In [ ]:
# ===== Dataloaders + training =====
train_ds = FFPPDualStreamDataset(train_df, frames_per_video=CFG['frames_per_video'], frame_size=CFG['frame_size'], segformer_extractor=seg_uncert)
val_ds = FFPPDualStreamDataset(val_df, frames_per_video=CFG['frames_per_video'], frame_size=CFG['frame_size'], segformer_extractor=seg_uncert)
test_ds = FFPPDualStreamDataset(test_df, frames_per_video=CFG['frames_per_video'], frame_size=CFG['frame_size'], segformer_extractor=seg_uncert)
train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True, num_workers=CFG['num_workers'], pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=CFG['batch_size'], shuffle=False, num_workers=CFG['num_workers'], pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=CFG['batch_size'], shuffle=False, num_workers=CFG['num_workers'], pin_memory=True)
model = DualStreamDeepfakeNet().to(DEVICE)
model, hist_df, best = train_one_run(model, train_loader, val_loader, CFG, device=DEVICE)
print('Best epoch:', best['epoch'])
print('Best val metrics:', json.dumps(best['metrics'], indent=2))

In [ ]:
# ===== Final test metrics + plots =====
test_metrics, y_true, y_prob = evaluate_model(model, test_loader, device=DEVICE)
print('Test metrics:')
print(json.dumps(test_metrics, indent=2))
if len(hist_df) > 0:
    fig, ax = plt.subplots(1, 2, figsize=(12,4))
    ax[0].plot(hist_df['epoch'], hist_df['train_loss'], marker='o')
    ax[0].set_title('Train Loss')
    ax[0].grid(alpha=0.3)
    ax[1].plot(hist_df['epoch'], hist_df['f1'], marker='o', label='val_f1')
    if hist_df['roc_auc'].notna().any():
        ax[1].plot(hist_df['epoch'], hist_df['roc_auc'], marker='o', label='val_auc')
    ax[1].set_title('Validation Metrics')
    ax[1].legend()
    ax[1].grid(alpha=0.3)
    plt.show()

In [ ]:
# ===== Save .pth bundle (project-compatible, no .pkl) =====
save_path = CFG['save_dir'] / CFG['model_name']
bundle = {
    'model_state_dict': model.state_dict(),
    'model_class': 'DualStreamDeepfakeNet',
    'config': {k:(str(v) if isinstance(v, Path) else v) for k,v in CFG.items()},
    'best_val': best,
    'test_metrics': test_metrics,
    'label_map': {0:'real', 1:'deepfake'},
    'decision_threshold': 0.5
}
torch.save(bundle, save_path)
print('Saved:', save_path)

In [ ]:
# ===== Single-video inference helper =====
@torch.no_grad()
def predict_video_binary(model, video_path, threshold=0.5):
    temp_df = pd.DataFrame([{'video_path': str(video_path), 'label': 0, 'label_name': 'unknown'}])
    ds = FFPPDualStreamDataset(temp_df, frames_per_video=CFG['frames_per_video'], frame_size=CFG['frame_size'], segformer_extractor=seg_uncert)
    batch = ds[0]
    xg = batch['x_global'].unsqueeze(0).to(DEVICE)
    xl = batch['x_local'].unsqueeze(0).to(DEVICE)
    prob = torch.sigmoid(model(xg, xl)).item()
    return {
        'video_path': str(video_path),
        'deepfake_probability': float(prob),
        'prediction': 'deepfake' if prob >= threshold else 'real'
    }
# Example:
# print(predict_video_binary(model, '/kaggle/input/.../video.mp4'))